<a href="https://colab.research.google.com/github/umiSirya/callcentre_performance/blob/main/call_centre_data_analysis%26cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [30]:
dp = pd.read_csv('call_centre_metrics_dataset.csv', sep=';')
dp.head(10)

,agent_id,date,product_id,lang_id,calls_handled,avg_aht,std_pass
0,10,2020-07-13,1,2,29,"754,75",1
1,6,2020-07-15,2,2,18,"432,03",1
2,10,2020-07-04,1,2,32,"1.325,75",0
3,4,2020-07-23,2,2,13,"545,04",1
4,2,2020-07-20,3,2,6,"86,19",0
5,9,2020-07-07,3,1,39,"266,03",1
6,3,2020-01-17,1,1,12,"1.765,72",0
7,7,2020-07-07,1,1,28,"1.728,45",0
8,2,2020-07-16,2,2,6,"598,08",0
9,4,2020-07-11,2,2,10,"75,46",0


In [31]:
dp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   agent_id       270 non-null    int64 
 1   date           270 non-null    object
 2   product_id     270 non-null    int64 
 3   lang_id        270 non-null    int64 
 4   calls_handled  270 non-null    int64 
 5   avg_aht        270 non-null    object
 6   std_pass       270 non-null    int64 
dtypes: int64(5), object(2)
memory usage: 14.9+ KB


In [32]:
#checking for missing values
dp.isna().sum()

,0
agent_id,0
date,0
product_id,0
lang_id,0
calls_handled,0
avg_aht,0
std_pass,0


In [33]:
#checking on avg_aht since it loaded as text
dp['avg_aht'].head(5)

,avg_aht
0,"754,75"
1,"432,03"
2,"1.325,75"
3,"545,04"
4,"86,19"


In [34]:
#basic statistics
dp.describe()

,agent_id,product_id,lang_id,calls_handled,std_pass
count,270.000000,270.000000,270.000000,270.000000,270.000000
mean,5.500000,2.000000,1.500000,19.992593,0.607407
std,2.877615,0.818013,0.500929,10.364363,0.489234
min,1.000000,1.000000,1.000000,1.000000,0.000000
25%,3.000000,1.000000,1.000000,11.000000,0.000000
50%,5.500000,2.000000,1.500000,20.000000,1.000000
75%,8.000000,3.000000,2.000000,29.000000,1.000000
max,10.000000,3.000000,2.000000,39.000000,1.000000


# **Data Cleaning**

In [35]:
#renaming dataset
df= dp.copy()

In [36]:
# Fix avg_aht to standard float (European format → standard)
df['avg_aht'] = (df['avg_aht']
                 .str.replace('.', '', regex=False)   # remove thousands separator
                 .str.replace(',', '.', regex=False)  # swap decimal comma to point
                 .astype(float))                      # convert text to number

# Convert date from string to datetime
df['date'] = pd.to_datetime(df['date'])

# Verify fixes
print(df.dtypes)
print(df[['date', 'avg_aht']].head(5))

agent_id                  int64
date             datetime64[ns]
product_id                int64
lang_id                   int64
calls_handled             int64
avg_aht                 float64
std_pass                  int64
dtype: object
        date  avg_aht
0 2020-07-13   754.75
1 2020-07-15   432.03
2 2020-07-04  1325.75
3 2020-07-23   545.04
4 2020-07-20    86.19


In [42]:
# duplicate rows
df.duplicated().sum()

np.int64(0)

# **EDA**

In [38]:
# How many unique agents, products and languages?
print(df['agent_id'].nunique())
print(df['product_id'].nunique())
print(df['lang_id'].nunique())
print("Date range", df['date'].min(), df['date'].max())

10
3
2
Date range 2020-01-01 00:00:00 2020-07-27 00:00:00


In [39]:
# What does avg_aht look like across the dataset?
print(df['avg_aht'].describe())


count     270.000000
mean      691.167333
std       568.868008
min         8.080000
25%       209.067500
50%       510.955000
75%       973.752500
max      2992.150000
Name: avg_aht, dtype: float64


In [40]:
# how many calls each agent handled in total?
print(df.groupby('agent_id')['calls_handled'].sum().sort_values(ascending=False))

agent_id
9     966
10    886
7     759
8     695
5     580
6     496
3     401
4     311
1     188
2     116
Name: calls_handled, dtype: int64


In [41]:
# overall pass rate
print("\nOverall pass rate:")
print(df['std_pass'].value_counts())
print(f"Pass rate: {df['std_pass'].mean()*100:.1f}%")


Overall pass rate:
std_pass
1    164
0    106
Name: count, dtype: int64
Pass rate: 60.7%


In [43]:
# checking for outliers since minimum AHT was 8 seconds
print('Suspiciously low AHT (under 30 seconds):')
print(df[df['avg_aht'] < 30])

Suspiciously low AHT (under 30 seconds):
     agent_id       date  product_id  lang_id  calls_handled  avg_aht  \
45          7 2020-07-03           3        1             31    24.15   
96          4 2020-07-09           3        2             14    18.43   
192         8 2020-07-26           3        2             29    25.71   
193         2 2020-07-04           2        2              5     8.08   
251         9 2020-07-19           3        1             37    22.01   
258         2 2020-07-08           3        2              7     9.52   

     std_pass  
45          1  
96          1  
192         1  
193         0  
251         1  
258         0  


In [44]:
# Flag Low AHT
df['low_aht_flag'] = (df['avg_aht'] < 10).astype(int)

print("Flagged rows:", df['low_aht_flag'].sum())
print("\nFlagged records:")
print(df[df['low_aht_flag'] == 1])


Flagged rows: 2

Flagged records:
     agent_id       date  product_id  lang_id  calls_handled  avg_aht  \
193         2 2020-07-04           2        2              5     8.08   
258         2 2020-07-08           3        2              7     9.52   

     std_pass  low_aht_flag  
193         0             1  
258         0             1  


In [45]:
# Export cleaned data to CSV
df.to_csv('callcenter_clean.csv', index=False)
